task 2 is to build a function that calculates whether the trade of buying gas in summer (cheaper) and selling it in winter(expensive) is profitable or not and by how much.

this is called pricing the contract



we need get_prices from task 1 so we can look up at gas prices on any date

In [1]:
import pandas as pd
import numpy as np
from scipy.interpolate import CubicSpline

# load data and rebuild price function from task 1
df = pd.read_csv('Nat_Gas.csv')
df['Dates'] = pd.to_datetime(df['Dates'], format='%m/%d/%y')
df['days'] = (df['Dates'] - df['Dates'].min()).dt.days
df['month'] = df['Dates'].dt.month

spline = CubicSpline(df['days'], df['Prices'])      # fit cubic spline through all 48 data points for smooth interpolation
monthly_avg = df.groupby('month')['Prices'].mean()

def get_price(date_str):
    date = pd.to_datetime(date_str)
    days = (date - df['Dates'].min()).days
    max_days = (df['Dates'].max() - df['Dates'].min()).days

    if days <= max_days:
        price = spline(days)
    
    else:
        price = monthly_avg[date.month]

    return round(float(price), 2)

print("Price function ready")
print("Jun 2022:", get_price('2022-06-15'))
print("Jan 2025:", get_price('2025-01-15'))

Price function ready
Jun 2022: 10.55
Jan 2025: 11.78


Contract Price Model:

buy gas at a given date -> pay price (injection)

sell gas at a given date -> recieve price (withdraw)

profit = selling price - buying price

subtract costs -> injection cost, withdraw cost, monthly storage cost

In [2]:
def price_contract(injection_date, withdraw_date, injection_rate, withdraw_rate, max_storage, storage_cost_per_month):

    contract_value = 0

    for inj_date, with_date in zip(injection_date, withdraw_date):

        # get prices
        buy_price = get_price(inj_date)
        sell_price = get_price(with_date)

        # volume -> amount of natural gas being stored or traded
        volume = min(injection_rate, withdraw_rate, max_storage)

        # revenue and cost
        revenue = sell_price * volume
        buy_cost = buy_price * volume

        # storage duration in months
        inj = pd.to_datetime(inj_date)
        withd = pd.to_datetime(with_date)
        months_stored = (withd.year - inj.year) * 12 + (withd.month - inj.month)

        total_storage_cost = storage_cost_per_month * months_stored

        #  contract value
        contract_value += revenue - buy_cost - total_storage_cost

    return round(contract_value, 2)

# Test it
result = price_contract(
    injection_date = ['2022-06-01', '2022-03-01', '2023-01-01'],
    withdraw_date = ['2022-12-01', '2022-09-01', '2023-07-01'],
    injection_rate = 1000000,
    withdraw_rate = 1000000,
    max_storage = 1000000,
    storage_cost_per_month = 100000)

print(f"Contract Value: ${result:,.2f}")

Contract Value: $-2,980,000.00


In [6]:
result = price_contract(
    injection_date=['2021-06-01', '2022-06-01', '2023-06-01'],
    withdraw_date=['2021-12-01', '2022-12-01', '2023-12-01'],
    injection_rate=1000000,
    withdraw_rate=1000000,
    max_storage=1000000,
    storage_cost_per_month=100000
)

print(f"Contract Value: ${result:,.2f}")

Contract Value: $1,530,000.00
